# 01 — Explore and verify the raw dataset

This notebook starts the data-understanding phase. First, it fingerprints the raw file, then checks its basic structure and recorded values. It does not train a model.

> **Integrity note:** The project's expected checksum prefix is `4ad2333067fcb0a9`, but the supplied file was observed to have prefix `1e0d557990cd7533`. The current file is used only under the user's explicit checksum override; the mismatch is shown below and is not treated as a successful match.

## 1. Locate the repository and import the project loader

Finding the project root from the current working directory makes the notebook work whether Jupyter starts in the repository root or in `notebooks/`. The project loader checks the schema and value ranges.

In [ ]:
from pathlib import Path
import sys

candidates = [Path.cwd(), *Path.cwd().parents]
ROOT = next(
    (candidate for candidate in candidates
     if (candidate / "data" / "raw" / "lung_cancer_combined_22457.csv").is_file()),
    None,
)
if ROOT is None:
    raise FileNotFoundError(
        "Could not locate data/raw/lung_cancer_combined_22457.csv. "
        "Start Jupyter from the repository or one of its subfolders."
    )

if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

from src.config import EXPECTED, RAW_FILE
from src.data_io import load_raw, sha256

print(f"Repository root: {ROOT}")
print(f"Raw data file: {RAW_FILE.relative_to(ROOT)}")

## 2. Verify the file fingerprint

A SHA-256 digest acts as a file fingerprint. A matching prefix is evidence that the file bytes match the expected snapshot; a mismatch must be reported, not silently replaced with the expected value. Per the explicit project override, this notebook continues with the present file after displaying the discrepancy.

In [ ]:
actual_sha256 = sha256(RAW_FILE)
actual_prefix = actual_sha256[:16]
expected_prefix = EXPECTED["sha256_prefix"]
checksum_matches = actual_prefix == expected_prefix

print(f"Expected SHA-256 prefix: {expected_prefix}")
print(f"Actual SHA-256 prefix:   {actual_prefix}")
print(f"Full SHA-256:            {actual_sha256}")
print(f"Prefix matches:          {checksum_matches}")

if not checksum_matches:
    print("WARNING: checksum mismatch. Continuing only under the approved override.")

## 3. Load and validate the raw data

`verify=False` is intentional here because the checksum override was explicitly approved. Schema validation still runs. The mismatch remains a documented limitation of this data snapshot.

In [ ]:
df = load_raw(path=RAW_FILE, verify=False)

print(f"Shape: {df.shape}")
print(f"Total missing values: {int(df.isna().sum().sum())}")
print(f"patient_id is unique: {df['patient_id'].is_unique}")
print("\nColumns:")
print(df.columns.tolist())
print("\nDtypes:")
display(df.dtypes.rename("dtype").to_frame())

symptom_columns = df.columns[3:-1]
for column in symptom_columns:
    print(column, sorted(df[column].unique()))

print("GENDER_CODE values:", sorted(df["GENDER"].unique()))
display(df["AGE"].describe())

## 4. Inspect labels, age, and coded values

These summaries describe the contents of this file only. The documented meaning of the label and the mapping of `GENDER` codes remain unverified.

In [ ]:
import pandas as pd

target_counts = df["LUNG_CANCER"].value_counts().sort_index().rename_axis("recorded_label")
target_summary = target_counts.to_frame("count")
target_summary["share"] = target_summary["count"] / len(df)
display(target_summary)

display(df["AGE"].describe().to_frame().T)

coded_columns = [
    "GENDER", "SMOKING", "YELLOW_FINGERS", "ANXIETY", "PEER_PRESSURE",
    "CHRONIC_DISEASE", "FATIGUE", "ALLERGY", "WHEEZING",
    "ALCOHOL_CONSUMING", "COUGHING", "SHORTNESS_OF_BREATH",
    "SWALLOWING_DIFFICULTY", "CHEST_PAIN", "LUNG_CANCER",
]
value_sets = {column: sorted(df[column].unique().tolist()) for column in coded_columns}
display(pd.Series(value_sets, name="observed_values").to_frame())

## 5. Recorded-label balance and a trivial accuracy benchmark

The fraction of rows with recorded label `1` is also the accuracy a rule that always returns `1` would obtain on this same file. This is arithmetic on the observed labels, not a trained or evaluated model. It shows why raw accuracy is not a meaningful success criterion for this imbalanced file: a trivial rule receives a high score without using any features.

In [ ]:
negative_count = int(df["LUNG_CANCER"].eq(0).sum())
positive_count = int(df["LUNG_CANCER"].eq(1).sum())
positive_prevalence = float(df["LUNG_CANCER"].eq(1).mean())
always_positive_accuracy = positive_prevalence

print(f"Recorded label 0: {negative_count:,} ({negative_count / len(df):.4%})")
print(f"Recorded label 1: {positive_count:,} ({positive_prevalence:.4%})")
print(f"Always-positive rule accuracy on this file: {always_positive_accuracy:.4%}")
print("This is a trivial reference calculation, not a trained model or a success metric.")

### Interpretation and population limitation

The file's recorded-label-1 prevalence is high. An always-positive rule would therefore look strong under accuracy while failing to distinguish the two recorded-label groups. Do not use accuracy alone as a success metric. The sampling frame and label meaning are unknown, so this file's prevalence cannot be assumed to represent a real screening population; that population-level generalizability limitation remains OPEN. This project does not train a model.

## 6. Audit row order and the leading block

If early rows contain only one recorded label, row position itself can reveal that label. We identify the first positive row by its zero-based index, check that every earlier row is negative, and compare binary feature rates among leading-block negatives and later negatives. The comparison is descriptive evidence of a distribution shift in this file, not proof of how the file was produced.

In [ ]:
from src.audit import leading_block_end, plot_order_audit, source_shift

first_positive_index = leading_block_end(df)
leading_block = df.iloc[:first_positive_index]
print(f"First positive label index (0-based): {first_positive_index}")
print(f"All preceding labels are negative: {leading_block['LUNG_CANCER'].eq(0).all()}")
print(f"Leading-block rows: {len(leading_block)}")

smoking_lead = leading_block["SMOKING"].eq(2).mean()
later_negatives = df.iloc[first_positive_index:]
later_negatives = later_negatives[later_negatives["LUNG_CANCER"].eq(0)]
smoking_later_neg = later_negatives["SMOKING"].eq(2).mean()
print(f"SMOKING = 2 share, leading negatives: {smoking_lead:.4f}")
print(f"SMOKING = 2 share, later negatives:  {smoking_later_neg:.4f}")

source_shift_table = source_shift(df, first_positive_index)
source_shift_table.to_csv(ROOT / "reports" / "tables" / "source_shift.csv", index=False)
display(source_shift_table)

order_rolling = plot_order_audit(df, window=300)
print(f"Saved figure: {ROOT / 'reports' / 'figures' / 'f1_order.png'}")
print(f"Saved rolling-rate table: {ROOT / 'reports' / 'tables' / 'f1_order_rolling.csv'}")

### Interpretation

The rolling recorded-label rate stays at zero through the leading block, then rises after the first positive row. Compare smoking and `GENDER` code-1 rolling rates in the same plot. The feature-rate differences between leading-block negatives and later negatives show that the block differs in composition; describing it as a possible source shift is a hypothesis, because dataset provenance is unknown.

## 7. Audit exact duplicates and conflicting labels

Repeated feature patterns can make rows non-independent, while the same feature pattern with both recorded labels is a label conflict. The identifier is excluded from duplicate definitions because it is not a substantive feature. We also compare observed later-row duplicates with a seeded simulation that independently samples each feature's observed marginal distribution.

In [ ]:
from src.audit import duplicate_report, expected_duplicates_under_independence

duplicate_summary = duplicate_report(df, first_positive_index)
for name, value in duplicate_summary.items():
    print(f"{name}: {value}")

feature_columns = [column for column in df.columns if column not in {"patient_id", "LUNG_CANCER"}]
later_rows = df.iloc[first_positive_index:]
expected_later_duplicates = expected_duplicates_under_independence(
    later_rows, reps=10, seed=42
)
observed_later_duplicates = int(later_rows.duplicated(subset=feature_columns).sum())
print(f"Expected later-row duplicates under independent marginals (10 simulations): {expected_later_duplicates:.1f}")
print(f"Observed later-row duplicates on features: {observed_later_duplicates}")

### Interpretation

An exact duplicate is defined by matching the 15 features and recorded label, while a feature-only duplicate ignores the label. A conflicting pattern has identical feature values but both label values. The independent-marginal simulation is only a reference for the number of repeated patterns under that constructed null; it does not establish the real data-generation process or whether duplicate rows correspond to distinct people.

## 8. Check the identifier for order association

`patient_id` is sequential in this file, so any association with the recorded label may reflect row ordering rather than a meaningful patient characteristic. We calculate its rank-based AUC as a leakage diagnostic only; it must not enter an analysis feature set.

In [ ]:
from src.audit import auc_of_id

patient_id_auc = auc_of_id(df)
print(f"AUC of patient_id alone versus recorded label: {patient_id_auc:.4f}")
print("patient_id is an identifier/order field and is excluded from features.")
print(f"Source-shift Cramer's V range: {source_shift_table['cramers_v'].min():.4f} to {source_shift_table['cramers_v'].max():.4f}")
print(f"Largest source-shift p-value: {source_shift_table['p_value'].max():.3e}")

The ID-only AUC is substantially above 0.5 in this file, consistent with association between row order and the recorded label. This does not make the identifier a valid feature; instead, it is a leakage warning and reinforces excluding IDs and row position from all feature sets.

## 9. Check whether later rows have independent-looking feature structure

After restricting to rows from the first positive index onward and removing duplicate feature-plus-label records, we summarize correlations, pairwise chi-square tests among the 14 binary fields, variance inflation factors (VIF), and the standardized-feature PCA spectrum. These are diagnostics of structure in this file; they cannot establish provenance.

In [ ]:
from src.audit import independence_report

feature_columns = [column for column in df.columns if column not in {"patient_id", "LUNG_CANCER"}]
bulk = df.iloc[first_positive_index:].drop_duplicates(subset=feature_columns + ["LUNG_CANCER"])
independence_summary = independence_report(bulk)
for name, value in independence_summary.items():
    print(f"{name}: {value}")

pd.DataFrame([independence_summary]).to_csv(
    ROOT / "reports" / "tables" / "independence_report.csv", index=False
)
print(f"Saved report: {ROOT / 'reports' / 'tables' / 'independence_report.csv'}")

### Interpretation

A small maximum correlation, a pairwise significance share near 5%, VIF values near 1, and a near-flat PCA spectrum indicate little detectable feature redundancy in these later rows. This pattern is consistent with independent-like or resampled structure, but it does not identify the file's source or establish how it was created.

## 10. Apply and compare the four cleaning views

Cleaning policies are explicit analysis choices. View B removes exact duplicates and is primary; C additionally removes feature patterns with conflicting labels; D removes the leading raw-index block; A is the unmodified raw view for artifact comparison. The raw data file is not modified.

In [ ]:
from src.clean import make_views, summarize, write_cleaning_outputs

views = make_views(df)
view_summary = summarize(views)
display(view_summary)

treatment_summary = write_cleaning_outputs(df, views)
display(treatment_summary)
print(f"Saved views to: {ROOT / 'data' / 'interim'}")
print(f"Saved treatments: {ROOT / 'reports' / 'tables' / 'treatments.csv'}")
print(f"Saved age flags: {ROOT / 'reports' / 'tables' / 'age_iqr_flags.csv'}")

The view sizes and recorded-label rates quantify sensitivity to cleaning choices. D is formed by filtering B on original row index, not by dropping the first 1,455 rows from the already deduplicated table; this preserves the intended leading-block boundary. AGE outliers are flagged and retained.

## 11. H1: smoking code and recorded label

H1 null hypothesis: the recorded-label proportion is equal for rows coded `SMOKING = 2` and `SMOKING = 1`. We report a Pearson chi-square test without continuity correction, an odds ratio with Wald 95% CI, risk difference with a 95% CI, Cramer's V, and a seeded permutation p-value. These are associations in this file, not causal effects.

In [ ]:
from src.hypotheses import h1_smoking

h1_results = h1_smoking(views, n_perms=5000, seed=42)
display(h1_results)
h1_results.to_csv(ROOT / "reports" / "tables" / "h1_smoking.csv", index=False)
print(f"Saved H1 table: {ROOT / 'reports' / 'tables' / 'h1_smoking.csv'}")

### Interpretation

Compare the raw association with the deduplicated and leading-block sensitivity views. The odds ratio expresses the ratio of odds between smoking-code groups; the interval expresses uncertainty. A small p-value alone does not measure effect size or the probability that the null is true. The analysis is limited to this file and does not establish a real-world effect.

## 12. H4: a 14-feature family with Benjamini–Hochberg adjustment

Testing many features raises the chance of false positives. Benjamini–Hochberg adjusts p-values to control the false discovery rate across the pre-specified family. Here the family comprises `GENDER` code and the 13 symptom/lifestyle flags; age is analyzed separately.

In [ ]:
from src.stats_utils import benjamini_hochberg, feature_family

bh_check = benjamini_hochberg([0.01, 0.04, 0.03, 0.005])
print("BH hand-check:", bh_check)
binary_feature_family = ["GENDER"] + [
    column for column in df.columns[3:-1]
]
family_results = feature_family(views["D_dedup_nolead"], binary_feature_family)
display(family_results.sort_values("p").round(4))
family_results.to_csv(ROOT / "reports" / "tables" / "feature_family_D.csv", index=False)
print(f"D smallest p: {family_results['p'].min():.4f}")
print(f"D smallest q_BH: {family_results['q_BH'].min():.4f}")
print(f"D largest Cramer's V: {family_results['v'].max():.4f}")
print(f"Saved family table: {ROOT / 'reports' / 'tables' / 'feature_family_D.csv'}")

The adjusted q-values address the fact that this is a family of 14 tests. Report both effect sizes and adjusted significance; do not interpret nominal p-values in isolation. A failure to reject within this file does not establish that a feature has no relationship in other populations.

## 13. H2–H5: age, respiratory count, feature family, and integrity shift

H2 compares age ranks between recorded-label groups; CLES is the probability that a randomly selected positive-label row has a higher age than a negative-label row, with half credit for ties. H3 checks Spearman association between a four-item respiratory symptom count and label. H4 repeats the adjusted 14-feature family over all views. H5 summarizes leading-block versus later-negative feature differences.

In [ ]:
from src.hypotheses import h2_age, h3_respiratory_count

h2_results = h2_age(views, bootstrap_reps=300, seed=42)
h3_results = h3_respiratory_count(views, bootstrap_reps=200, seed=42)
display(h2_results)
display(h3_results)

family_by_view = pd.concat(
    [
        feature_family(view, binary_feature_family).assign(view=name)
        for name, view in views.items()
    ],
    ignore_index=True,
)
display(family_by_view.groupby("view").agg(
    minimum_p=("p", "min"), minimum_q_BH=("q_BH", "min"), maximum_V=("v", "max")
))
display(source_shift_table[["feature", "cramers_v", "p_value"]])

resp_count_distribution_B = (
    views["B_dedup"]["SMOKING"].to_frame()
)
resp_count_distribution_B["RESP_SYMPTOM_COUNT"] = views["B_dedup"][
    ["COUGHING", "WHEEZING", "SHORTNESS_OF_BREATH", "CHEST_PAIN"]
].eq(2).sum(axis=1)
print("View B respiratory-count distribution:",
      resp_count_distribution_B["RESP_SYMPTOM_COUNT"].value_counts().sort_index().to_dict())

h2_results.to_csv(ROOT / "reports" / "tables" / "h2_age.csv", index=False)
h3_results.to_csv(ROOT / "reports" / "tables" / "h3_respiratory_count.csv", index=False)
family_by_view.to_csv(ROOT / "reports" / "tables" / "feature_family_all_views.csv", index=False)
print("Saved H2, H3, and all-view H4 tables.")

### Interpretation

The raw view and the deduplicated views can show nominal associations that are very small in magnitude. Compare effect sizes and uncertainty across views, especially D, which removes the leading block. Statistical significance is sample- and policy-dependent; none of these tests establish causation, verified label semantics, or generalizability beyond this file.

## 14. Build the data and statistical visualization set

Each registered figure answers one question and is saved as a 200-dpi PNG with a one-line takeaway and, where applicable, a CSV table of plotted values. The suite contains data and statistical visualizations only; the optional model comparison is excluded.

In [ ]:
import json
from src.figures import build_guide_figures
from src.hypotheses import run_hypothesis_suite

hypothesis_results = run_hypothesis_suite(df, views, n_perms=5000, bootstrap_reps=300, seed=42)
figure_tables = build_guide_figures(df, views, hypothesis_results)
expected_figure_files = sorted(json.loads((ROOT / "reports" / "figures" / "captions.json").read_text(encoding="utf-8")))
for figure_name in expected_figure_files:
    figure_path = ROOT / "reports" / "figures" / figure_name
    print(figure_name, figure_path.is_file(), figure_path.stat().st_size if figure_path.is_file() else None)
print(f"Registered figures: {len(expected_figure_files)}")

## 15. Engineer and verify descriptive feature sets

The raw symptom codes are converted to 0/1, `GENDER` is renamed `GENDER_CODE`, and counts and age indicators are derived from input fields only. The recorded label is retained separately, while the identifier is removed. The scaler below is fitted on view B for descriptive outputs only; any future model must fit scaling on its training folds.

In [ ]:
from src.features import FEATURE_SETS, build_feature_outputs, verify

engineered_B, age_scaler_B, feature_tables, screening_table, resp_count_table = build_feature_outputs(
    views["B_dedup"]
)
print("AGE scaler fitted on B (descriptive only):", age_scaler_B)
print("Feature set sizes:", {name: len(columns) for name, columns in FEATURE_SETS.items()})
print("verify:", verify(engineered_B))
print("RESP_SYMPTOM_COUNT distribution:",
      engineered_B["RESP_SYMPTOM_COUNT"].value_counts().sort_index().to_dict())
print("Output tables:", sorted(path.name for path in (ROOT / "data" / "processed").glob("features_B_*.csv")))
display(screening_table.sort_values("p").head(10))

The three feature sets are alternative descriptive specifications, not trained models. The screening table does not drive feature selection. Because associations in D are negligible and the source is uncertain, F2 is recorded as prior-driven rather than evidence-selected. `SMOKER_IN_SCREEN_AGE` is only a proxy and cannot establish screening eligibility without smoking dose and quit-time information.

## Optional leakage experiment — intentionally not run

The guide suggests fitting logistic regression in cross-validation as an optional demonstration. This project’s governing scope excludes predictive-model fitting, so the experiment is not executed or included in the deliverable. Duplicate and row-order leakage are instead demonstrated with the direct integrity audits above. Any later model evaluation requires separate approval and leakage-safe design.

## Takeaway

Record the observed checksum and mismatch status; do not claim the file matches the expected fingerprint. The shape, dtypes, missingness, label distribution, age summary, coded-value sets, and trivial accuracy benchmark above are runtime outputs and should be used as the evidence for the next audit step.

## Generated visualization gallery

Run `python -m src.run_analysis` followed by `python -m src.results` from the project root before executing this cell. The figures and their questions/takeaways come from the generated project outputs; no predictive model is fitted.

In [ ]:
import json
from IPython.display import Image, Markdown, display

results = json.loads((ROOT / "docs" / "results.json").read_text(encoding="utf-8"))
for filename in results["figure_files"]:
    details = results["figures"][filename]
    display(Markdown(f"**{details['question']}**  \n{details['takeaway']}"))
    display(Image(filename=str(ROOT / "reports" / "figures" / filename)))